## Step 1 — Install dependencies (run once)

In [15]:
!pip install -q langchain_text_splitters sentence_transformers faiss-cpu transformers torch torchvision torchaudio pypdf python-docx


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
!pip install -q ddgs


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
!pip install -U transformers accelerate -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from ddgs import DDGS

def web_search(query: str, max_results: int = 3) -> str:

    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No web results found."
        return "\n\n".join(
            f"[{r['title']}]({r['href']})\n{r.get('body', '')}" for r in results
        )
    except Exception as e:
        return f"Web search failed: {e}"

print(web_search("Central bank of kenya monetary policy rate"))



[CBK | Central Bank of Kenya](https://www.centralbank.go.ke/)
The Central Bank of Kenya (CBK) celebrated its 50 years of existence and service to the nation of Kenya in September 2016. These celebrations entailed the CBK’s involvement in a number of activities. ... The Monetary Policy Committee retained the Central Bank Rate (CBR) at 8.75 percent at its ...

[Monetary Policy | CBK](https://www.centralbank.go.ke/category/monetary-policy/)
Monetary PolicyThe Monetary Policy Committee retains the Central Bank Rate (CBR) at 8.75 percent at its June 9, 2026...

[Kenya central bank holds key rate, monitors impact of oil prices on inflation | CNBC Africa](https://www.cnbcafrica.com/2026/kenyas-central-bank-maintains-main-lending-rate-at-8-75)
June 10, 2026 - NAIROBI, June 9 (Reuters) – Kenya’s central bank kept its benchmark lending rate at 8.75% on Tuesday and said it would monitor the impact of global oil prices on inflation. On Wednesday, Central Bank Governor Kamau Thugge will address a .

In [19]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "llama3.2:1b"

def safe_invoke(prompt_text, max_new_tokens=300):
    try:
        response = requests.post(OLLAMA_URL, json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt_text}],
            "stream": False,
            "options": {"num_predict": max_new_tokens},
        }, timeout=60)
        response.raise_for_status()
        return response.json()["message"]["content"]
    except Exception as e:
        return f"[Ollama error: {e}]"

print(safe_invoke("Say hello in one sentence."))

I'm happy to help you with anything you need.


In [20]:
!pip install -U accelerate


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
import numpy as np
import json

In [22]:
import sys
!{sys.executable} -m pip install -U accelerate


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Load all models (run once)
Embedder, re-ranker, and the LLM all load here. This is the only slow step — do it once per session, then never touch it again.

In [23]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss, numpy as np, json

print("Loading embedder...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Loading re-ranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Loading LLM (this is the slow part)...")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

print(f"\nAll models loaded. LLM running on: {model.device}")

Loading embedder...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4158.77it/s]


Loading re-ranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4418.87it/s]


Loading LLM (this is the slow part)...


c:\Users\admin\Desktop\hello\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 107.54it/s]



All models loaded. LLM running on: cpu


## Function definitions (run once, right after Step 2)
Nothing to edit here — just run it. This defines everything Steps 3 and 4 rely on.

In [24]:
current_index = None
current_chunks = None
current_file_name = None

CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

def extract_text(file_path):
    if file_path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        reader = PdfReader(file_path)
        return "\n".join(page.extract_text() for page in reader.pages)
    elif file_path.lower().endswith(".docx"):
        from docx import Document
        doc = Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs)
    elif file_path.lower().endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError("Unsupported file type — use .pdf, .docx, or .txt")

def load_document(file_path):
    global current_index, current_chunks, current_file_name

    print("1. Extracting text...")
    text = extract_text(file_path)

    print("2. Extracted text length:", len(text))

    if not text.strip():
        raise ValueError(
            "PDF se koi text extract nahi hua. "
            "PDF scanned/image-based ho sakti hai."
        )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    new_chunks = splitter.split_text(text)

    print("3. Number of chunks:", len(new_chunks))

    if not new_chunks:
        raise ValueError("Document se chunks generate nahi hue.")

    print("4. Generating embeddings...")

    new_embeddings = embedder.encode(
        new_chunks,
        show_progress_bar=True
    )

    print("5. Embedding type:", type(new_embeddings))
    print("6. Embedding shape:", getattr(new_embeddings, "shape", None))

    if len(new_embeddings) == 0:
        raise ValueError("Embeddings generate nahi hue.")

    if len(new_embeddings.shape) != 2:
        raise ValueError(
            f"Unexpected embedding shape: {new_embeddings.shape}"
        )

    new_index = faiss.IndexFlatL2(new_embeddings.shape[1])

    new_index.add(np.asarray(new_embeddings, dtype="float32"))

    current_index = new_index
    current_chunks = new_chunks
    current_file_name = file_path

    print(
        f"\n'{file_path}' loaded — "
        f"{len(new_chunks)} chunks indexed and ready."
    )

def retrieve(query, top_k=3, fetch_k=10):
    if current_index is None:
        raise RuntimeError("No document loaded yet — run Step 3 first.")
    q_emb = embedder.encode([query])
    _, idx = current_index.search(np.array(q_emb), min(fetch_k, len(current_chunks)))
    candidates = [current_chunks[i] for i in idx[0]]
    pairs = [[query, c] for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked[:top_k]]

def safe_invoke(prompt_text, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def rag_answer(question, top_k=3, verbose=True):
    if current_index is None:
        print("⚠️ No document loaded yet — run Step 3 to upload a file first.")
        return None
    retrieved_chunks = retrieve(question, top_k=top_k)
    context = "\n\n---\n\n".join(retrieved_chunks)
    prompt = (
        "You are a training content assistant. Answer using ONLY the context below.\n"
        "If the answer is not in the context, say so explicitly. "
        "ALWAYS return the output in JSON format with keys 'answer' and 'found_in_context' (true/false).\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    answer = safe_invoke(prompt)

    found = True
    try:
      parsed = json.loads(answer[answer.find("{"):answer.rfind("}")+1])
      found = parsed.get("found_in_context", True)
    except Exception:
      pass


    if not found:
      if verbose:
        print(" Not Found in Document - Searching back to web search...")
      web_results = web_search(question)
      web_prompt = (
          f"Answer this question using web search result below, "
          f"Mention that this result came from the web search, not from the uploaded document"
          f"Web Result: \n{web_results}\n\nQuestion: {question}"
      )
      answer = safe_invoke(web_prompt)
      source = "Web"
    else:
      source = "document"


    if verbose:
        print(f"[Document: {current_file_name}]")
        print(f"Q: {question}")

        print(f"A: {answer}\n")
        print(f"Source used: {source}")
    return answer

print("Functions ready.")

Functions ready.


## Step 3 — Load a document
Run this whenever you want to switch to a new or updated file. Re-running this cell replaces the previously loaded document — questions in Step 4 will then answer from the new file.

In [25]:
file_path = r"C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf"

print("Loading:", file_path)

load_document(file_path)

Ignoring wrong pointing object 333 0 (offset 0)
Ignoring wrong pointing object 335 0 (offset 0)
Ignoring wrong pointing object 342 0 (offset 0)
Ignoring wrong pointing object 384 0 (offset 0)
Ignoring wrong pointing object 752 0 (offset 0)


Loading: C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf
1. Extracting text...
2. Extracted text length: 108421
3. Number of chunks: 260
4. Generating embeddings...


Batches: 100%|██████████| 9/9 [00:04<00:00,  2.10it/s]

5. Embedding type: <class 'numpy.ndarray'>
6. Embedding shape: (260, 384)

'C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf' loaded — 260 chunks indexed and ready.


## Step 4 — Ask questions
Run this cell as many times as you like, in any order. Fast every time — no model reloading, no re-uploading. Always answers from whichever document was most recently loaded in Step 3.

In [26]:
my_question = "What is Claude Code?"  # ← edit this line, then re-run the cell
rag_answer(my_question)

 Not Found in Document - Searching back to web search...
[Document: C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf]
Q: What is Claude Code?
A: Claude Code is an agentic coding tool developed by Anthropic. It is designed to understand and interact with a developer's codebase, allowing users to edit files, run commands, and integrate with their development environment. The tool is available in various platforms including the terminal, integrated development environments (IDEs), desktop applications, and browsers. Claude Code aims to enhance productivity by providing assistance across multiple tasks typically performed through the command line.

Source used: Web


"Claude Code is an agentic coding tool developed by Anthropic. It is designed to understand and interact with a developer's codebase, allowing users to edit files, run commands, and integrate with their development environment. The tool is available in various platforms including the terminal, integrated development environments (IDEs), desktop applications, and browsers. Claude Code aims to enhance productivity by providing assistance across multiple tasks typically performed through the command line."

### (Optional) Ask several questions at once

In [27]:
questions = [
    "What is Claude Code?",
    "How do you install Claude Code?",
]
for q in questions:
    rag_answer(q)

 Not Found in Document - Searching back to web search...
[Document: C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf]
Q: What is Claude Code?
A: Claude Code is an AI coding agent developed by Anthropic. It integrates with various platforms such as GitHub, GitLab, and command-line tools like the terminal. The purpose of Claude Code is to streamline the entire software development process, allowing users to read issues, write code, run tests, and submit pull requests all within their terminal environment. This integration helps users avoid switching between different tools and workflows, making it easier to manage projects and collaborate effectively.

Source used: Web
 Not Found in Document - Searching back to web search...
[Document: C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf]
Q: How do you install Claude Code?
A: To install Claude Code, follow these steps based on the information provided in the web search results:

1. **Download the D

### (Optional) Live input loop — good for a room full of people

In [28]:
while True:
    q = input("Ask a question (or 'quit' to stop): ").strip()
    if not q or q.lower() == "quit":
        print("Stopped.")
        break
    rag_answer(q)

 Not Found in Document - Searching back to web search...
[Document: C:\Users\admin\Downloads\introduction-to-data-science-using-python.pdf]
Q: can u give me 10 interview questions about python ?
A: Sure! Below are ten Python interview questions:

### 1. **Write a function to reverse a string without using Python's slicing [::-1] or reversed()**.
   ```python
   def reverse_string(s):
       return s[::-1]
   
   print(reverse_string("Hello World"))  # Output: "dlroW olleH"
   ```

### 2. **Given two strings, write a function to check if they are anagrams of each other**.
   ```python
   def is_anagram(str1, str2):
       return sorted(str1) == sorted(str2)

   print(is_anagram("listen", "silent"))  # Output: True
   ```

### 3. **Implement a simple calculator that performs basic arithmetic operations (+, -, *, /)**.
   ```python
   def calculate(expression):
       stack = []
       tokens = expression.split()
       
       for token in tokens:
           if token.isdigit():
         